In [1]:
import base64
import os
import sys
import re

from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import snowflake.connector

from cryptography.hazmat.primitives import serialization
from dotenv import dotenv_values
from pyspark.sql import SparkSession, Window, functions as F

In [ ]:
# Airflow supplies this date. For a manual notebook run, None uses today.
partition_date = None


In [2]:
# Use the date supplied by Airflow, or today's date for a manual notebook run.
processing_timezone = "Asia/Kolkata"
if partition_date is None:
    partition_date = datetime.now(
        ZoneInfo(processing_timezone)
    ).strftime("%d%m%Y")

partition_date = datetime.strptime(partition_date, "%d%m%Y").strftime("%d%m%Y")

# Configure local Spark.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault(
    "JAVA_HOME",
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
)
driver_memory = os.environ.get("SKYPOINTS_SPARK_DRIVER_MEMORY", "512m")
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-memory {driver_memory} pyspark-shell"

# Docker uses the connector JARs downloaded when its image was built.
# Local notebook runs retain the existing Maven package configuration.
spark_jars_dir = os.environ.get("SKYPOINTS_SPARK_JARS")
spark_jars = (
    ",".join(str(path) for path in sorted(Path(spark_jars_dir).glob("*.jar")))
    if spark_jars_dir else ""
)
if spark_jars_dir and not spark_jars:
    raise RuntimeError("No connector JARs were found in SKYPOINTS_SPARK_JARS.")



spark = (
    SparkSession.builder
    .appName("SkyPointsGold")
    .master(os.environ.get("SKYPOINTS_SPARK_MASTER", "local[1]"))
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", processing_timezone)
    .config("spark.jars", spark_jars)
    .config(
        "spark.jars.packages",
        "" if spark_jars else (
            "net.snowflake:spark-snowflake_2.12:3.2.0-spark_3.5,"
            "net.snowflake:snowflake-jdbc:4.0.2"
        ),
    )
    .getOrCreate()
)


# Read the existing credentials.
project_root = Path(
    os.environ.get(
        "SKYPOINTS_PROJECT_ROOT",
        '/Users/mskbaruah/code/Incubyte Take home assignment/skypoints-pipeline',
    )
)

sf = {
    **dotenv_values(project_root / ".env.local", interpolate=False),
    **os.environ,
}


# Prepare the private key for authentication.
passphrase = sf.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")

private_key = serialization.load_pem_private_key(
    Path(
        sf.get("SNOWFLAKE_LOCAL_PRIVATE_KEY_FILE")
        or sf["SNOWFLAKE_PRIVATE_KEY_FILE"]
    ).expanduser().read_bytes(),
    password=passphrase.encode() if passphrase else None,
)

key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)


# Spark will read member data from STAGING.
# Keep this dictionary private because it contains authentication material.
sf_options = {
    "sfURL": f"{sf['SNOWFLAKE_ACCOUNT']}.snowflakecomputing.com",
    "sfUser": sf["SNOWFLAKE_USER"],
    "sfRole": sf["SNOWFLAKE_ROLE"],
    "sfWarehouse": sf["SNOWFLAKE_WAREHOUSE"],
    "sfDatabase": sf["SNOWFLAKE_DATABASE"],
    "sfSchema": "STAGING",
    "pem_private_key": base64.b64encode(key_bytes).decode("ascii"),
}


# Native SQL connection for subsequent GOLD updates.
# Destination tables will be referenced explicitly as GOLD.<table_name>.
sf_connection = snowflake.connector.connect(
    account=sf["SNOWFLAKE_ACCOUNT"],
    user=sf["SNOWFLAKE_USER"],
    role=sf["SNOWFLAKE_ROLE"],
    warehouse=sf["SNOWFLAKE_WAREHOUSE"],
    database=sf["SNOWFLAKE_DATABASE"],
    schema="STAGING",
    authenticator="SNOWFLAKE_JWT",
    private_key=key_bytes,
    session_parameters={"TIMEZONE": "UTC"},
)


# Verify the Snowflake connection.
with sf_connection.cursor() as cursor:
    cursor.execute("SELECT 1")
    cursor.fetchone()

print("Connected to Snowflake.")
print(f"Processing partition: {partition_date}")

26/11/23 18:12:57 WARN Utils: Your hostname, Monisankars-MacBook-Air.local resolves to a loopback address: 127.0.2.3; using 192.168.1.8 instead (on interface en0)
26/11/23 18:12:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/mskbaruah/.ivy2/cache
The jars for the packages stored in: /Users/mskbaruah/.ivy2/jars
net.snowflake#spark-snowflake_2.12 added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1d936a24-e41e-405f-970f-ef15a2dc99de;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/mskbaruah/code/Incubyte%20Take%20home%20assignment/skypoints-pipeline/skypoints/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found net.snowflake#spark-snowflake_2.12;3.2.0-spark_3.5 in central
	found net.snowflake#snowflake-jdbc;4.0.2 in central
	found org.apache.parquet#parquet-avro;1.15.2 in central
	found org.apache.parquet#parquet-column;1.15.2 in central
	found org.apache.parquet#parquet-common;1.15.2 in central
	found org.apache.parquet#parquet-format-structures;1.15.2 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.slf4j#slf4j-api;1.7.33 in central
	found org.apache.parquet#parquet-encoding;1.15.2 in central
	found org.apache.parquet#parquet-hadoop;1.15.2 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.6-6 in central
	found org.apache.avro#avro;1.11.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.14.3 in central
	found com.fasterxml.jackson.core#jackson-databind;2.14.3 in central
	found com.fasterxml

Connected to Snowflake.
Processing partition: 23112026


In [3]:
# Validate the processing date.
partition_date = datetime.strptime(
    partition_date,
    "%d%m%Y",
).strftime("%d%m%Y")

# Read only the selected partition.
members_query = f"""
    SELECT *
    FROM STAGING.MEMBERS
    WHERE PARTITION_DATE = '{partition_date}'
"""

members_staging = (
    spark.read
    .format("net.snowflake.spark.snowflake")
    .options(**sf_options)
    .option("query", members_query)
    .load()
)

# Preview all columns.
print(f"Staging partition: {partition_date}")
members_staging.show(5, truncate=False)

26/11/23 18:13:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,015 ms, timeout: 1,000 ms
26/11/23 18:13:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,015 ms, timeout: 1,000 ms
26/11/23 18:13:06 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal HTTP/1.1
26/11/23 18:13:06 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email HTTP/1.1
26/11/23 18:13:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,419 ms, timeout: 1,000 ms
26/11/23 18:13:06 ERROR RestRequest: Returning null response. Cause: java.net.SocketTimeoutException: Connect timed out, request: P

Staging partition: 23112026


+------------------+---------+---------------+----------------+---------+------------+-----+-------+----------+---------+---------+--------------+---+------------+
|MEMBER_NAME       |MEMBER_ID|ENROLLMENT_DATE|LAST_FLIGHT_DATE|TIER_CODE|AGENT_NAME  |STATE|COUNTRY|DOB       |IS_ACTIVE|POST_CODE|PARTITION_DATE|AGE|STALE_MEMBER|
+------------------+---------+---------------+----------------+---------+------------+-----+-------+----------+---------+---------+--------------+---+------------+
|Monserrate Johns  |1        |2023-02-11     |2026-09-22      |SLV      |Olivia Hart |SA   |AUS    |1995-01-04|I        |NULL     |23112026      |31 |false       |
|Mindy Tillman     |10       |2021-11-28     |2026-02-27      |SLV      |Nikhil Hayes|IL   |USA    |2000-08-11|I        |NULL     |23112026      |26 |true        |
|Jaida N. W. Mraz  |100      |2023-11-29     |2026-08-26      |PLT      |Maya Arden  |IL   |USA    |1970-10-20|A        |NULL     |23112026      |56 |false       |
|Jaida L. J. Kuh

In [4]:
countries = members_staging.select("COUNTRY").distinct()

countries.orderBy("COUNTRY").show(100, truncate=False)

+-------+
|COUNTRY|
+-------+
|AUS    |
|BRA    |
|GBR    |
|IND    |
|PRT    |
|USA    |
+-------+



In [5]:
country_names = {
    "AUS": "Australia",
    "BRA": "Brazil",
    "GBR": "United Kingdom",
    "IND": "India",
    "PRT": "Portugal",
    "USA": "United States",
}

country_tables = {
    code: "TABLE_" + re.sub(r"[^A-Z0-9]+", "_", name.upper())
    for code, name in country_names.items()
}

country_lookup = spark.createDataFrame(
    [
        (code, name, country_tables[code])
        for code, name in country_names.items()
    ],
    "COUNTRY string, COUNTRY_NAME string, TARGET_TABLE string",
)

routing_preview = (
    members_staging
    .select("COUNTRY")
    .distinct()
    .join(F.broadcast(country_lookup), on="COUNTRY", how="left")
    .withColumn(
        "TARGET_TABLE",
        F.when(
            F.col("COUNTRY").isNull(),
            F.lit("TABLE_UNASSIGNED"),
        ).otherwise(F.col("TARGET_TABLE")),
    )
)

unknown_countries = routing_preview.filter(
    F.col("COUNTRY").isNotNull()
    & F.col("TARGET_TABLE").isNull()
)

if not unknown_countries.isEmpty():
    unknown_countries.show(truncate=False)
    raise ValueError("Add the missing countries to country_names.")

routing_preview.orderBy("COUNTRY").show(truncate=False)

+-------+--------------+--------------------+
|COUNTRY|COUNTRY_NAME  |TARGET_TABLE        |
+-------+--------------+--------------------+
|AUS    |Australia     |TABLE_AUSTRALIA     |
|BRA    |Brazil        |TABLE_BRAZIL        |
|GBR    |United Kingdom|TABLE_UNITED_KINGDOM|
|IND    |India         |TABLE_INDIA         |
|PRT    |Portugal      |TABLE_PORTUGAL      |
|USA    |United States |TABLE_UNITED_STATES |
+-------+--------------+--------------------+



In [6]:
load_partition = datetime.strptime(
    partition_date,
    "%d%m%Y",
).strftime("%d%m%Y")

work_table = "SKYPOINTS.GOLD._MEMBERS_WORK"

# USA layout, also used for countries without a separate template.
# Each entry maps a GOLD column to its corresponding STAGING column.
default_columns = {
    "ID": "MEMBER_ID",
    "Name": "MEMBER_NAME",
    "TierCode": "TIER_CODE",
    "EnrollmentDate": "ENROLLMENT_DATE",
    "FlightDate": "LAST_FLIGHT_DATE",
    "PARTITION_DATE": "PARTITION_DATE",
}

# India and Australia have their own column layouts.
country_columns = {
    "TABLE_INDIA": {
        "ID": "MEMBER_ID",
        "Name": "MEMBER_NAME",
        "DOB": "DOB",
        "TierCode": "TIER_CODE",
        "EnrollmentDate": "ENROLLMENT_DATE",
        "Flight Date": "LAST_FLIGHT_DATE",
        "PARTITION_DATE": "PARTITION_DATE",
    },
    "TABLE_AUSTRALIA": {
        "Unique ID": "MEMBER_ID",
        "Member Name": "MEMBER_NAME",
        "Tier Type": "TIER_CODE",
        "Date of Birth": "DOB",
        "Date of Enrollment": "ENROLLMENT_DATE",
        "Date of Flight": "LAST_FLIGHT_DATE",
        "PARTITION_DATE": "PARTITION_DATE",
    },
}

# Prepare the country routing rules used by Cell 7.
country_rules = []
country_parameters = []

for code, table_name in country_tables.items():
    country_rules.append("WHEN s.COUNTRY = %s THEN %s")
    country_parameters.extend((code, table_name))

country_case = "\n".join(country_rules)


def quote_column(name):
    """Handle column names containing spaces or mixed capitalization."""
    return '"' + name.replace('"', '""') + '"'


def build_country_merge_sql(table_name):
    """Build the merge statement using the destination country's layout."""
    column_map = country_columns.get(table_name, default_columns)

    key_column = next(
        target_name
        for target_name, source_name in column_map.items()
        if source_name == "MEMBER_ID"
    )

    selected_columns = ",\n".join(
        f"incoming.{quote_column(source_name)} "
        f"AS {quote_column(target_name)}"
        for target_name, source_name in column_map.items()
    )

    insert_columns = ", ".join(
        quote_column(name) for name in column_map
    )

    insert_values = ", ".join(
        f"source.{quote_column(name)}" for name in column_map
    )

    update_columns = [
        name for name in column_map
        if name != key_column
    ]

    update_assignments = ",\n".join(
        f"{quote_column(name)} = source.{quote_column(name)}"
        for name in update_columns
    )

    changed_condition = "\nOR ".join(
        f"target.{quote_column(name)} IS DISTINCT FROM "
        f"source.{quote_column(name)}"
        for name in update_columns
    )

    return f"""
        MERGE INTO IDENTIFIER(%s) AS target
        USING (
            SELECT
                incoming.MEMBER_ID AS _MEMBER_ID,
                incoming._TARGET_TABLE,
                {selected_columns}
            FROM {work_table} AS incoming
            WHERE incoming._TARGET_TABLE = %s
               OR incoming._PREVIOUS_TABLE = %s
        ) AS source
            ON target.{quote_column(key_column)} = source._MEMBER_ID

        WHEN MATCHED
            AND source._TARGET_TABLE <> %s
            THEN DELETE

        WHEN MATCHED
            AND ({changed_condition})
            THEN UPDATE SET
                {update_assignments}

        WHEN NOT MATCHED
            AND source._TARGET_TABLE = %s
            THEN INSERT ({insert_columns})
            VALUES ({insert_values})
    """

In [7]:
with sf_connection.cursor() as cursor:
    cursor.execute("ALTER SESSION SET LOCK_TIMEOUT = 60")

    # Create the empty working table outside the transaction.
    cursor.execute(f"""
        CREATE OR REPLACE TEMPORARY TABLE {work_table}
        DATA_RETENTION_TIME_IN_DAYS = 0
        AS
        SELECT
            s.*,
            CAST(NULL AS DATE) AS _RECORD_DATE,
            CAST(NULL AS VARCHAR(255)) AS _TARGET_TABLE,
            CAST(NULL AS VARCHAR(255)) AS _PREVIOUS_TABLE
        FROM SKYPOINTS.STAGING.MEMBERS AS s
        WHERE 1 = 0
    """)

    try:
        cursor.execute("BEGIN TRANSACTION")

        # Coordinate publication with other pipeline writers.
        cursor.execute("""
            UPDATE SKYPOINTS.RAW.INGESTION_LOCK
            SET LOCK_VERSION = LOCK_VERSION + 1
            WHERE LOCK_ID = 1
        """)

        if cursor.rowcount != 1:
            raise RuntimeError("Expected exactly one ingestion-lock row.")

        # Prepare eligible records once for this run.
        cursor.execute(
            f"""
            INSERT INTO {work_table}
            WITH incoming AS (
                SELECT
                    s.*,
                    TO_DATE(
                        s.PARTITION_DATE,
                        'DDMMYYYY'
                    ) AS _RECORD_DATE,
                    CASE
                        WHEN s.COUNTRY IS NULL
                            THEN 'TABLE_UNASSIGNED'
                        {country_case}
                    END AS _TARGET_TABLE
                FROM SKYPOINTS.STAGING.MEMBERS AS s
                WHERE s.PARTITION_DATE = %s
            )
            SELECT
                incoming.*,
                previous.TARGET_TABLE AS _PREVIOUS_TABLE
            FROM incoming
            LEFT JOIN SKYPOINTS.GOLD.MEMBER_LOCATION AS previous
                ON incoming.MEMBER_ID = previous.MEMBER_ID
            WHERE previous.MEMBER_ID IS NULL
               OR incoming._RECORD_DATE >= previous.RECORD_DATE
            """,
            tuple(country_parameters) + (load_partition,),
        )

        # Validate the prepared records.
        cursor.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT MEMBER_ID),
                COUNT_IF(_TARGET_TABLE IS NULL)
            FROM {work_table}
        """)

        member_count, unique_members, unmapped_rows = cursor.fetchone()

        if member_count != unique_members:
            raise ValueError(
                "The eligible batch contains duplicate or null member IDs."
            )

        if unmapped_rows:
            raise ValueError(
                "Some countries have no destination. Review country_names."
            )

        # Include both destination tables and previous country tables.
        cursor.execute(f"""
            SELECT _TARGET_TABLE AS TABLE_NAME
            FROM {work_table}

            UNION

            SELECT _PREVIOUS_TABLE AS TABLE_NAME
            FROM {work_table}
            WHERE _PREVIOUS_TABLE IS NOT NULL

            ORDER BY TABLE_NAME
        """)

        affected_tables = [
            row[0] for row in cursor.fetchall()
        ]

        # Use each country's column names and member-ID column.
        for table_name in affected_tables:
            cursor.execute(
                build_country_merge_sql(table_name),
                (
                    f"SKYPOINTS.GOLD.{table_name}",
                    table_name,
                    table_name,
                    table_name,
                    table_name,
                ),
            )

        # Update locations after all country writes succeed.
        cursor.execute(f"""
            MERGE INTO SKYPOINTS.GOLD.MEMBER_LOCATION AS target
            USING (
                SELECT
                    MEMBER_ID,
                    _TARGET_TABLE AS TARGET_TABLE,
                    _RECORD_DATE AS RECORD_DATE
                FROM {work_table}
            ) AS source
                ON target.MEMBER_ID = source.MEMBER_ID

            WHEN MATCHED
                AND (
                    target.RECORD_DATE < source.RECORD_DATE
                    OR target.TARGET_TABLE <> source.TARGET_TABLE
                )
                THEN UPDATE ALL BY NAME

            WHEN NOT MATCHED THEN
                INSERT ALL BY NAME
        """)

        # Clear the working data in the same transaction.
        cursor.execute(f"TRUNCATE TABLE {work_table}")

        sf_connection.commit()

    except BaseException:
        sf_connection.rollback()
        raise

print(f"Processing partition: {load_partition}")
print(f"Evaluated {member_count:,} eligible members.")
print(f"Processed {len(affected_tables)} country tables.")

Processing partition: 23112026
Evaluated 10,000 eligible members.
Processed 6 country tables.
